# [대조군] 원본 Enkidu 패치 균일 축소 — 보호본 만들기

**질문:** v1(마스킹)·v2(XTTS 대리 모델)는 STOI 0.87에서 정화 후 잔존이 0이었다. 원인이 **섭동 크기**인가, v1·v2가 **섭동 모양**을 바꾼 탓인가?

**방법:** 원본 Enkidu 패치(`runs/run1/enkidu_ufp`)를 재학습 없이 **배율 0.25**로만 줄여 붙인다. v2 걸러보기에서 같은 방식으로 붙였을 때 STOI 0.873이어서 v1(0.869), v2(0.872)와 명료도가 같다. 붙이는 방식은 v2와 같다(Wiener 필터 끔, 전역 타일링).

- 잔존 0 → 크기가 원인. 작은 잡음형 섭동은 모양과 무관하게 정화에 지워진다
- 잔존이 남음 → 원본 패치 모양에 정화를 버티는 요소가 있고, v1·v2가 그것을 망가뜨렸다

conda·XTTS 없이 기본 Colab에서 돈다(GPU 불필요). 출력: `xtts_exp/ctrl_exp/uniform_c025/D2`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pystoi

In [ ]:
import torch, torchaudio, soundfile as sf, numpy as np
from pathlib import Path
from pystoi import stoi

R = Path("/content/drive/MyDrive/xtts_exp")
C = 0.25                                   # 원본 Enkidu 패치 배율
NOISE_LEVEL, L, NF, HOP, SR = 0.4, 30, 1024, 512, 16000
SRC = R / "runs/run1/enkidu_ufp"           # 원본 패치 (읽기만)
OUT = R / "ctrl_exp/uniform_c025/D2"
assert OUT.resolve() != (R / "D2").resolve(), "기존 D2 경로입니다"
win = torch.hann_window(NF)

def apply(x, nr, ni, level):
    S = torch.stft(x, NF, HOP, NF, window=win, return_complex=True).unsqueeze(0)
    T = S.shape[-1]
    Dr, Di = torch.zeros_like(S.real), torch.zeros_like(S.imag)
    if L > T:
        Dr[:, :, :T] = nr[:, :, :T]; Di[:, :, :T] = ni[:, :, :T]
    else:
        for i in range(T // L):
            Dr[:, :, i*L:(i+1)*L] = nr; Di[:, :, i*L:(i+1)*L] = ni
    S2 = torch.complex(S.real + level * Dr, S.imag + level * Di)
    return torch.istft(S2, NF, HOP, NF, window=win, length=x.numel()).clamp(-1, 1).flatten()

stois, n = [], 0
spks = sorted(p.name for p in (R / "D1").iterdir() if p.is_dir() and not p.name.startswith("."))
for spk in spks:
    paths = sorted((R / "D1" / spk).glob("*.wav"))
    ev = paths[max(1, int(len(paths) * 0.7)):]             # 01 노트북과 같은 평가 분할
    assert {p.name for p in ev} == {p.name for p in (R / "D2" / spk).glob("*.wav")}, f"{spk}: 평가셋 불일치"
    d = torch.load(SRC / f"{spk}.pt", map_location="cpu")
    nr, ni = d["real"].float().cpu(), d["imag"].float().cpu()
    (OUT / spk).mkdir(parents=True, exist_ok=True)
    for p in ev:
        x, sr = sf.read(str(p), dtype="float32")
        if x.ndim > 1: x = x.mean(1)
        xt = torch.from_numpy(x)
        if sr != SR: xt = torchaudio.functional.resample(xt, sr, SR)
        y = apply(xt, nr, ni, NOISE_LEVEL * C).numpy()
        torchaudio.save(str(OUT / spk / p.name), torch.from_numpy(y).unsqueeze(0), SR)
        stois.append(stoi(xt.numpy(), y, SR, extended=False)); n += 1
    print(f"[{spk}] {len(ev)}개")

print(f"\n저장 {n}개 -> {OUT}")
print(f"평균 STOI {np.mean(stois):.3f}  (0.87 근처면 정상)")

## 다음

이 폴더의 `02_purify_ctrl.ipynb`(입력 `ctrl_exp/uniform_c025/D2`) → `03_clone_ctrl.ipynb`(`RUN_NAME = enkidu_uniform_c025`, C1·C4 재사용) 순서로 실행한다.